# LLC Additivity Analysis for CMSP

Interactive analysis and visualization of LLC additivity experiments.

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.additivity import compute_additivity_defect, compute_full_additivity_defect, summarize_results

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Load Results

In [ ]:
# Adjust this path to your results directory
results_dir = Path('../results/')

# Load all results if available
all_results_path = results_dir / 'all_results.pkl'
if all_results_path.exists():
    with open(all_results_path, 'rb') as f:
        all_results = pickle.load(f)
    print(f'Loaded results for seeds: {list(all_results.keys())}')
else:
    # Load single seed
    seed_dirs = sorted(results_dir.glob('seed_*'))
    all_results = {}
    for sd in seed_dirs:
        seed = int(sd.name.split('_')[1])
        llc_path = sd / 'llc_results.pkl'
        if llc_path.exists():
            with open(llc_path, 'rb') as f:
                all_results[seed] = {'llc_results': pickle.load(f)}
    print(f'Found results for seeds: {list(all_results.keys())}')

## 2. LLC Overview

In [ ]:
# Collect LLC values across seeds
llc_data = []
for seed, res in all_results.items():
    llc_res = res['llc_results']
    for subtask, vals in llc_res.items():
        llc_data.append({
            'seed': seed,
            'subtask': subtask,
            'llc_mean': vals['llc_mean'],
            'llc_std': vals['llc_std'],
        })

llc_df = pd.DataFrame(llc_data)
print(llc_df.groupby('subtask')[['llc_mean', 'llc_std']].agg(['mean', 'std']))

In [ ]:
# Bar plot of LLC per subtask (averaged across seeds)
agg = llc_df.groupby('subtask')['llc_mean'].agg(['mean', 'std']).sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(agg)), agg['mean'], yerr=agg['std'], capsize=5, alpha=0.8)
ax.set_xticks(range(len(agg)))
ax.set_xticklabels(agg.index, rotation=45, ha='right')
ax.set_ylabel('LLC (λ̂)')
ax.set_title('LLC per Subtask Data Restriction')
plt.tight_layout()
plt.show()

## 3. Loss Trace Diagnostics

Check SGLD mixing: loss traces should show random fluctuation, not monotone drift.

In [ ]:
# Pick first seed for trace diagnostics
seed0 = list(all_results.keys())[0]
llc_res = all_results[seed0]['llc_results']

subtasks_to_plot = list(llc_res.keys())[:4]  # first 4

fig, axes = plt.subplots(1, len(subtasks_to_plot), figsize=(5*len(subtasks_to_plot), 4))
if len(subtasks_to_plot) == 1:
    axes = [axes]

for ax, name in zip(axes, subtasks_to_plot):
    trace = llc_res[name].get('loss_trace')
    if trace is not None:
        for chain_trace in trace:
            ax.plot(chain_trace, alpha=0.5)
    ax.set_title(f'{name}')
    ax.set_xlabel('Draw')
    ax.set_ylabel('Loss')

plt.suptitle('SGLD Loss Traces per Chain')
plt.tight_layout()
plt.show()

## 4. Additivity Defects

In [ ]:
# Compute defects for each seed
defect_rows = []
for seed, res in all_results.items():
    llc_res = res['llc_results']
    available = set(llc_res.keys())
    
    # Find all valid pairwise triples
    atomics = sorted([k for k in available if k.startswith('{') and ',' not in k and '∪' not in k])
    for i in range(len(atomics)):
        for j in range(i+1, len(atomics)):
            union_key = f'{atomics[i]}∪{atomics[j]}'
            if union_key in available:
                delta = llc_res[atomics[i]]['llc_mean'] + llc_res[atomics[j]]['llc_mean'] - llc_res[union_key]['llc_mean']
                defect_rows.append({
                    'seed': seed,
                    'pair': f'{atomics[i]}+{atomics[j]}',
                    'delta': delta,
                })

if defect_rows:
    defect_df = pd.DataFrame(defect_rows)
    print(defect_df.groupby('pair')['delta'].agg(['mean', 'std']))
    
    # Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    agg = defect_df.groupby('pair')['delta'].agg(['mean', 'std'])
    colors = ['#2196F3' if m >= 0 else '#F44336' for m in agg['mean']]
    ax.bar(range(len(agg)), agg['mean'], yerr=agg['std'], capsize=5, color=colors, alpha=0.8)
    ax.set_xticks(range(len(agg)))
    ax.set_xticklabels(agg.index, rotation=45, ha='right')
    ax.set_ylabel('Additivity defect δ')
    ax.set_title('LLC Additivity Defects (Pairwise Atomics)')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    plt.tight_layout()
    plt.show()
else:
    print('No pairwise defects found — check that union dataloaders were created.')

## 5. Training Curves

In [ ]:
# Load training results for one seed
seed0 = list(all_results.keys())[0]
train_res_path = results_dir / f'seed_{seed0}' / 'results.pkl'
if train_res_path.exists():
    with open(train_res_path, 'rb') as f:
        train_res = pickle.load(f)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    eval_steps = train_res['eval_steps']
    for name, losses in train_res['subtask_losses'].items():
        ax.plot(eval_steps[:len(losses)], losses, label=name)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Per-Subtask Training Loss')
    ax.legend()
    ax.set_yscale('log')
    plt.tight_layout()
    plt.show()
else:
    print(f'No training results found at {train_res_path}')

## 6. Width Sweep Analysis

If you ran the width sweep experiment, load and compare defects across widths.

In [ ]:
# Placeholder for width sweep analysis
# Modify paths to match your experiment output
widths = [64, 128, 256, 512]
width_defects = []

for w in widths:
    path = results_dir / f'width_{w}' / 'all_results.pkl'
    if path.exists():
        with open(path, 'rb') as f:
            w_results = pickle.load(f)
        for seed, res in w_results.items():
            if 'full_defect' in res and res['full_defect'] is not None:
                width_defects.append({
                    'width': w,
                    'seed': seed,
                    'delta': res['full_defect']['delta'],
                })

if width_defects:
    wdf = pd.DataFrame(width_defects)
    agg = wdf.groupby('width')['delta'].agg(['mean', 'std'])
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(agg.index, agg['mean'], yerr=agg['std'], marker='o', capsize=5)
    ax.set_xlabel('Width')
    ax.set_ylabel('Full Additivity Defect δ')
    ax.set_title('LLC Additivity Defect vs Network Width')
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print('No width sweep results found.')